# 02 — Player & Squad Feature Engineering

Extracts **one row per (tournament_id, team_id)** from player-level club data.
Split into two sections:

**Section A — Player aggregates** (club performance → squad means by position)
- Shrinkage-adjusted per-appearance rates (goals, assists)
- Club Elo-weighted goal / assist volume
- Positional market-value proxied by `club_strength` (club Elo)
- GK clean-sheet rate

**Section B — Squad structure**
- Club cohesion: count of same-club player pairs (`squad_club_pairs`), largest block
- WC pedigree: prior tournament wins, finals appearances (from `worldcup.db`)
- Defending champion flag

**Input:**  `data/processed/player_stats_with_club_elo.csv`  
**Output:** `./data/features_player_squad.parquet`  
**Window:** 2006–2022 only (Transfermarkt data availability).

In [7]:
import warnings

warnings.filterwarnings("ignore")

import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd

BASE = Path("../../../../")  # repo root
PLAYER_PATH = BASE / "data/processed/player_stats_with_club_elo.csv"
DB_PATH = BASE / "data/raw/worldcup/data-sqlite/worldcup.db"
OUT_DIR = Path("./data")
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert PLAYER_PATH.is_file(), f"Player file not found: {PLAYER_PATH}"
assert DB_PATH.is_file(), f"DB not found: {DB_PATH}"
print("Paths OK")

Paths OK


In [8]:
df = pd.read_csv(PLAYER_PATH)

# Standardise column names used throughout this notebook
if "team_name" in df.columns and "national_team" not in df.columns:
    df = df.rename(columns={"team_name": "national_team"})
if "goals" in df.columns and "club_goals" not in df.columns:
    df = df.rename(columns={"goals": "club_goals"})

print(f"Raw player file: {df.shape}")
print(f"WC years: {sorted(df['wc_year'].unique())}")
print(f"perf_found rate: {df['perf_found'].mean():.1%}")

Raw player file: (2890, 56)
WC years: [np.int64(2006), np.int64(2010), np.int64(2014), np.int64(2018), np.int64(2022)]
perf_found rate: 92.2%


In [9]:
# Keep only players with club stats
active = df[df["perf_found"] == True].copy()

# Per-appearance rates (raw)
active["goals_per_app"] = active["club_goals"] / active["appearances"].replace(
    0, np.nan
)
active["assists_per_app"] = active["assists"] / active["appearances"].replace(0, np.nan)

print(f"Active (perf_found=True): {active.shape}")
print(f"Position distribution:\n{active['position_code'].value_counts()}")

Active (perf_found=True): (2664, 58)
Position distribution:
position_code
MF    887
DF    876
FW    608
GK    293
Name: count, dtype: int64


## A — Player aggregates per team × tournament

In [10]:
# ── Shrinkage-adjusted rates ───────────────────────────────────────────────
# Blends observed rate toward positional median when appearances < MIN_APP.
# Players at or above MIN_APP keep their observed rate.
MIN_APP = 10

for rate_col, count_col in [
    ("goals_per_app", "club_goals"),
    ("assists_per_app", "assists"),
]:
    pos_median = active.groupby("position_code")[rate_col].transform("median")
    w_obs = active["appearances"].clip(upper=MIN_APP)
    w_prior = MIN_APP - w_obs
    adj_col = rate_col + "_adj"
    active[adj_col] = (
        w_obs * active[rate_col].fillna(0) + w_prior * pos_median.fillna(0)
    ) / MIN_APP
    print(f"{adj_col}: {active[adj_col].describe().to_dict()}")

goals_per_app_adj: {'count': 2664.0, 'mean': 0.13378892622303484, 'std': 0.16671651095008785, 'min': 0.0, '25%': 0.0024999999999999996, '50%': 0.07317073170731707, '75%': 0.19047619047619047, 'max': 1.0869565217391304}
assists_per_app_adj: {'count': 2664.0, 'mean': 0.09371788411397894, 'std': 0.1004926775015277, 'min': 0.0, '25%': 0.0, '50%': 0.06521739130434782, '75%': 0.14583333333333334, 'max': 0.719047619047619}


In [11]:
# ── Elo-weighted goal / assist volume ──────────────────────────────────────
# Weight player contribution by club_strength (normalised club Elo).
mean_strength = active["club_strength"].mean()
active["elo_weight"] = active["club_strength"].fillna(mean_strength) / mean_strength
active["goals_elo_wt"] = active["club_goals"] * active["elo_weight"]
active["assists_elo_wt"] = active["assists"] * active["elo_weight"]

print("Elo-weighted stats computed.")

Elo-weighted stats computed.


In [12]:
def wmean(group, col, w_col="appearances"):
    """Appearance-weighted mean, returns NaN if group is empty."""
    g = group.dropna(subset=[col, w_col])
    ws = g[w_col].clip(lower=0)
    if ws.sum() == 0 or len(g) == 0:
        return np.nan
    return float(np.average(g[col], weights=ws))


player_rows = []
for (wc_tid, nat_team), grp in active.groupby(["tournament_id", "national_team"]):
    row = {"tournament_id": wc_tid, "national_team": nat_team}

    # Positional club strength (proxy for market value)
    for pos in ["FW", "MF", "DF"]:
        pg = grp[grp["position_code"] == pos]
        row[f"pl_{pos}_avg_strength"] = (
            pg["club_strength"].mean() if len(pg) > 0 else np.nan
        )

    # Squad-level adjusted rates (appearance-weighted)
    row["pl_squad_goals_adj"] = wmean(grp, "goals_per_app_adj")
    row["pl_squad_assists_adj"] = wmean(grp, "assists_per_app_adj")

    # Positional rates
    fw = grp[grp["position_code"] == "FW"]
    mf = grp[grp["position_code"] == "MF"]
    row["pl_FW_goals_adj"] = wmean(fw, "goals_per_app_adj") if len(fw) > 0 else np.nan
    row["pl_MF_assists_adj"] = (
        wmean(mf, "assists_per_app_adj") if len(mf) > 0 else np.nan
    )

    # Elo-weighted volume (sum over squad)
    row["pl_squad_goals_elo_wt"] = grp["goals_elo_wt"].sum()
    row["pl_squad_assists_elo_wt"] = grp["assists_elo_wt"].sum()
    row["pl_FW_goals_elo_wt"] = grp[grp["position_code"] == "FW"]["goals_elo_wt"].sum()
    row["pl_MF_assists_elo_wt"] = grp[grp["position_code"] == "MF"][
        "assists_elo_wt"
    ].sum()

    # Average club strength over squad
    row["pl_squad_avg_strength"] = grp["club_strength"].mean()

    # Offensive / defensive composites
    off = grp[grp["position_code"].isin(["FW", "MF"])]
    row["pl_offensive_score"] = (
        (
            wmean(off, "goals_per_app_adj") + wmean(off, "assists_per_app_adj")
            if len(off) > 0
            else np.nan
        )
        if not (wmean(off, "goals_per_app_adj") is np.nan)
        else np.nan
    )

    # GK clean-sheet rate
    gk = grp[grp["position_code"] == "GK"]
    if len(gk) > 0 and "clean_sheets" in gk.columns and "appearances" in gk.columns:
        total_app = gk["appearances"].sum()
        row["pl_GK_clean_sheet_rate"] = (
            gk["clean_sheets"].sum() / total_app if total_app > 0 else np.nan
        )
    else:
        row["pl_GK_clean_sheet_rate"] = np.nan

    # Club league position (lower = better finishing club)
    if "club_league_position" in grp.columns:
        row["pl_avg_club_position"] = grp["club_league_position"].mean()

    player_rows.append(row)

player_agg = pd.DataFrame(player_rows)
print(f"player_agg shape: {player_agg.shape}")
print(f"Columns: {list(player_agg.columns)}")

player_agg shape: (158, 16)
Columns: ['tournament_id', 'national_team', 'pl_FW_avg_strength', 'pl_MF_avg_strength', 'pl_DF_avg_strength', 'pl_squad_goals_adj', 'pl_squad_assists_adj', 'pl_FW_goals_adj', 'pl_MF_assists_adj', 'pl_squad_goals_elo_wt', 'pl_squad_assists_elo_wt', 'pl_FW_goals_elo_wt', 'pl_MF_assists_elo_wt', 'pl_squad_avg_strength', 'pl_offensive_score', 'pl_GK_clean_sheet_rate']


## B — Squad structure features

### B1 — Club cohesion (same-club player pairs)

In [13]:
# Count n*(n-1)/2 pairs for each club represented in the squad
cohesion_rows = []
club_col = "club_name" if "club_name" in df.columns else None

if club_col:
    for (wc_tid, nat_team), grp in df.groupby(["tournament_id", "national_team"]):
        grp_valid = grp[grp[club_col].notna() & (grp[club_col] != "")]
        club_counts = grp_valid.groupby(club_col).size()
        pairs = int((club_counts * (club_counts - 1) / 2).sum())
        largest = int(club_counts.max()) if len(club_counts) > 0 else 0
        cohesion_rows.append(
            {
                "tournament_id": wc_tid,
                "national_team": nat_team,
                "sq_club_pairs": pairs,
                "sq_largest_club_block": largest,
            }
        )
    cohesion_df = pd.DataFrame(cohesion_rows)
    print(f"cohesion_df shape: {cohesion_df.shape}")
else:
    print("club_name column not found — skipping cohesion features")
    cohesion_df = pd.DataFrame()

cohesion_df shape: (158, 4)


### B2 — WC pedigree & defending champion (from worldcup.db)

In [14]:
con = sqlite3.connect(DB_PATH)


def _sql(q):
    return pd.read_sql_query(q, con)


ta_all = _sql("SELECT * FROM team_appearances")
tourn_all = _sql("SELECT tournament_id, tournament_name, year FROM tournaments")
matches_all = _sql("SELECT match_id, tournament_id, stage_name FROM matches")
con.close()

# Men's WC only, add year
men_ids = tourn_all.loc[
    ~tourn_all["tournament_name"].str.contains("Women", case=False, na=False),
    "tournament_id",
].unique()
ta_men = ta_all[ta_all["tournament_id"].isin(men_ids)].merge(
    tourn_all[["tournament_id", "year"]], on="tournament_id", how="left"
)

# WC winners per year (team that won the final)
finals = matches_all[
    matches_all["stage_name"].str.lower().str.contains("final", na=False)
]
finals = finals[
    ~finals["stage_name"].str.lower().str.contains("semi|third|quarter", na=False)
]
final_winners = ta_men[
    ta_men["match_id"].isin(finals["match_id"]) & (ta_men["win"] == 1)
][["tournament_id", "team_id", "year"]].drop_duplicates()

# Map: in tournament Y, who was the defending champion (winner of Y-4)?
DEFENDING = {
    2006: "T-11",  # Brazil won 2002
    2010: "T-53",  # Italy won 2006
    2014: "T-52",  # Spain won 2010
    2018: "T-33",  # Germany won 2014
    2022: "T-32",  # France won 2018
}
# Fallback: derive from final_winners if available
year_to_winner = final_winners.set_index("year")["team_id"].to_dict()
DEFENDING_DERIVED = {y + 4: tid for y, tid in year_to_winner.items()}
DEFENDING_MAP = {**DEFENDING_DERIVED, **DEFENDING}  # manual overrides take precedence

print("Defending champion map:", DEFENDING_MAP)

Defending champion map: {1934: 'T-84', 1938: 'T-41', 1942: 'T-41', 1954: 'T-74', 1958: 'T-86', 1962: 'T-09', 1966: 'T-09', 1970: 'T-28', 1974: 'T-09', 1978: 'T-86', 1982: 'T-03', 1986: 'T-41', 1990: 'T-03', 1994: 'T-86', 1998: 'T-09', 2002: 'T-30', 2006: 'T-11', 2010: 'T-53', 2014: 'T-52', 2018: 'T-33', 2022: 'T-32', 2026: 'T-03'}


In [15]:
# Pedigree: for each (tournament, team), compute stats from ALL PRIOR men's WCs
# Stages: final, semi-finals for counting purposes
stage_order = {
    "group stage": 0,
    "round of 16": 1,
    "round of sixteen": 1,
    "quarter-finals": 2,
    "quarter-final": 2,
    "semi-finals": 3,
    "semi-final": 3,
    "third-place match": 3,
    "third place match": 3,
    "final": 4,
}
ta_men_stage = ta_men.merge(
    matches_all[["match_id", "stage_name"]], on="match_id", how="left"
)
ta_men_stage["stage_val"] = (
    ta_men_stage["stage_name"].str.lower().map(stage_order).fillna(0)
)

# Tournament-level team reach (max stage per team per tournament)
team_reach = (
    ta_men_stage.groupby(["tournament_id", "team_id", "year"])["stage_val"]
    .max()
    .reset_index(name="best_stage")
)
# Wins in the tournament
team_wins = (
    ta_men.groupby(["tournament_id", "team_id", "year"])["win"]
    .sum()
    .reset_index(name="tourn_wins")
)
team_tourn_hist = team_reach.merge(team_wins, on=["tournament_id", "team_id", "year"])

# Pedigree per (target_tournament, team)
wc_year_map = {
    "WC-2006": 2006,
    "WC-2010": 2010,
    "WC-2014": 2014,
    "WC-2018": 2018,
    "WC-2022": 2022,
}
pedigree_rows = []
all_team_ids = team_tourn_hist["team_id"].unique()
target_tourns = [(t, y) for t, y in wc_year_map.items()]

for wc_tid, wc_year in target_tourns:
    for tid in all_team_ids:
        prior = team_tourn_hist[
            (team_tourn_hist["team_id"] == tid) & (team_tourn_hist["year"] < wc_year)
        ]
        pedigree_rows.append(
            dict(
                tournament_id=wc_tid,
                team_id=tid,
                sq_wc_appearances=len(prior),
                sq_wc_total_wins=int(prior["tourn_wins"].sum()),
                sq_wc_finals=(prior["best_stage"] == 4).sum(),
                sq_wc_semis=(prior["best_stage"] >= 3).sum(),
                sq_defending_champion=int(DEFENDING_MAP.get(wc_year) == tid),
            )
        )

pedigree_df = pd.DataFrame(pedigree_rows)
print(f"pedigree_df shape: {pedigree_df.shape}")
print(
    "Defending champions flagged:",
    pedigree_df[pedigree_df["sq_defending_champion"] == 1][
        ["tournament_id", "team_id"]
    ].to_dict("records"),
)

pedigree_df shape: (425, 7)
Defending champions flagged: [{'tournament_id': 'WC-2006', 'team_id': 'T-11'}, {'tournament_id': 'WC-2010', 'team_id': 'T-53'}, {'tournament_id': 'WC-2014', 'team_id': 'T-52'}, {'tournament_id': 'WC-2018', 'team_id': 'T-33'}, {'tournament_id': 'WC-2022', 'team_id': 'T-32'}]


## C — Merge & save

In [16]:
# player_agg uses 'national_team' name; pedigree/cohesion use 'team_id'
# We need a mapping: (tournament_id, national_team) → team_id
# Use the player file's team_id column if available, otherwise join via team_name from DB
if "team_id" in df.columns:
    name_to_id = df[["tournament_id", "national_team", "team_id"]].drop_duplicates()
else:
    # Join via team_name lookup from DB
    con = sqlite3.connect(DB_PATH)
    teams_db = pd.read_sql_query("SELECT team_id, team_name FROM teams", con)
    con.close()
    name_to_id = player_agg[["tournament_id", "national_team"]].drop_duplicates()
    name_to_id = name_to_id.merge(
        teams_db.rename(columns={"team_name": "national_team"}),
        on="national_team",
        how="left",
    )

player_agg = player_agg.merge(
    name_to_id, on=["tournament_id", "national_team"], how="left"
)

# Merge all sections
feats = player_agg.copy()
if not cohesion_df.empty:
    feats = feats.merge(
        cohesion_df.rename(columns={"national_team": "national_team"}),
        on=["tournament_id", "national_team"],
        how="left",
    )
feats = feats.merge(pedigree_df, on=["tournament_id", "team_id"], how="left")

# Add wc_year for convenience
feats["year"] = feats["tournament_id"].map(wc_year_map)

out_path = OUT_DIR / "features_player_squad.parquet"
feats.to_parquet(out_path, index=False)
print(f"Saved → {out_path}")
print(f"Shape : {feats.shape}")
print(f"\nColumn list:")
for c in feats.columns:
    print(f"  {c}")

Saved → data/features_player_squad.parquet
Shape : (158, 25)

Column list:
  tournament_id
  national_team
  pl_FW_avg_strength
  pl_MF_avg_strength
  pl_DF_avg_strength
  pl_squad_goals_adj
  pl_squad_assists_adj
  pl_FW_goals_adj
  pl_MF_assists_adj
  pl_squad_goals_elo_wt
  pl_squad_assists_elo_wt
  pl_FW_goals_elo_wt
  pl_MF_assists_elo_wt
  pl_squad_avg_strength
  pl_offensive_score
  pl_GK_clean_sheet_rate
  team_id
  sq_club_pairs
  sq_largest_club_block
  sq_wc_appearances
  sq_wc_total_wins
  sq_wc_finals
  sq_wc_semis
  sq_defending_champion
  year


In [17]:
# Sanity checks
print("Missing rates:")
miss = feats.isnull().mean().sort_values(ascending=False)
print(miss[miss > 0].to_string())

print("\nSample (WC-2022, first 5):")
display(
    feats[feats["tournament_id"] == "WC-2022"][
        [
            "national_team",
            "pl_FW_avg_strength",
            "pl_squad_goals_adj",
            "sq_wc_total_wins",
            "sq_defending_champion",
        ]
    ].head()
)

Missing rates:
pl_GK_clean_sheet_rate    0.158228
pl_DF_avg_strength        0.088608
pl_FW_avg_strength        0.075949
pl_MF_avg_strength        0.050633
pl_MF_assists_adj         0.018987
pl_squad_avg_strength     0.018987
pl_FW_goals_adj           0.012658
pl_offensive_score        0.006329

Sample (WC-2022, first 5):


,national_team,pl_FW_avg_strength,pl_squad_goals_adj,sq_wc_total_wins,sq_defending_champion
126,Argentina,1841.225952,0.129071,47,0
127,Australia,1390.151683,0.144686,2,0
128,Belgium,1781.796543,0.167861,21,0
129,Brazil,1880.773167,0.158991,76,0
130,Cameroon,1756.716345,0.112321,4,0
